# 01 - Generate synthetic expense data

### This notebook creates the dataset the whole system is tested on: 100 reports, about 330 claims, 25% of them out of policy, with hidden answer-key labels.

In [1]:
import random
from datetime import date, timedelta
from pathlib import Path
from typing import Any

import pandas as pd
import yaml


def load_policy(path: Path) -> dict[str, Any]:
    """Load the expense policy config from a YAML file."""
    if not path.exists():
        raise FileNotFoundError(f"Policy file not found: {path}")
    with open(path) as f:
        return yaml.safe_load(f)


ROOT = Path.cwd().parent
SEED = 42
OUT_OF_POLICY_RATE = 0.25
BORDERLINE_RATE = 0.05
WINDOW_START = date(2026, 5, 1)
WINDOW_END = date(2026, 10, 31)
TODAY = date(2026, 11, 12)
rng = random.Random(SEED)
POLICY = load_policy(ROOT / "config" / "policy.yaml")
CATEGORIES = POLICY["categories"]
HIGH_COST_CITIES = set(POLICY["city_tiers"]["high_cost"])

assert POLICY["version"] == "1.0"
assert CATEGORIES["Lodging"]["nightly_cap"]["standard"] == 250.0
assert len(HIGH_COST_CITIES) == 6
assert random.Random(SEED).random() == random.Random(SEED).random()
print(f"Policy v{POLICY['version']} loaded: {len(CATEGORIES)} categories")

Policy v1.0 loaded: 10 categories


In [2]:
EMPLOYEES_PATH = ROOT / "data" / "raw" / "employees.csv"
TRAVEL_PROB = {
    "Sales": 0.7,
    "Marketing": 0.5,
    "Engineering": 0.3,
    "Operations": 0.4,
    "Finance": 0.3,
}


def load_employees(path: Path) -> pd.DataFrame:
    """Load employee roster, keeping every column as a string."""
    if not path.exists():
        raise FileNotFoundError(f"Employees file not found: {path}")
    return pd.read_csv(path, dtype=str)


employees_df = load_employees(EMPLOYEES_PATH)

assert len(employees_df) == 10, f"Expected 10 employees, got {len(employees_df)}"
assert employees_df["employee_id"].is_unique, "Duplicate employee_id found"

unknown_depts = set(employees_df["department"]) - set(TRAVEL_PROB)
assert not unknown_depts, f"Departments missing from TRAVEL_PROB: {unknown_depts}"

valid_managers = set(employees_df["employee_id"]) | {"BOARD"}
bad_managers = set(employees_df["manager_id"]) - valid_managers
assert not bad_managers, f"Unknown manager_id values: {bad_managers}"

print(f"Loaded: {len(employees_df)} employees")
employees_df.head(3)

Loaded: 10 employees


,employee_id,employee_name,title,department,home_city,manager_id
0,E001,Priya Raman,Director of Engineering,Engineering,"Austin, TX",E010
1,E002,Kevin Ortiz,Senior Software Engineer,Engineering,"Austin, TX",E001
2,E003,Lena Brooks,ML Engineer,Engineering,"Denver, CO",E001


In [3]:
MILEAGE_RATES = [
    (
        date.fromisoformat(band["from"]),
        date.fromisoformat(band["to"]),
        band["rate"],
    )
    for band in CATEGORIES["Mileage"]["rates"]
]


def money(amount: float) -> float:
    """Round a dollar amount to cents."""
    return round(amount, 2)


def rand_date(start: date, end: date, rand: random.Random) -> date:
    """Return a random date between start and end, inclusive.

    Args:
        start: First allowed date.
        end: Last allowed date.
        rand: Seeded random generator to draw from.

    Returns:
        A date in the closed range [start, end].
    """
    if end < start:
        raise ValueError(f"end {end} is before start {start}")
    span_days = (end - start).days
    return start + timedelta(days=rand.randint(0, span_days))


def city_tier(city: str) -> str:
    """Return the lodging cap tier for a city (POL-2.2)."""
    return "high_cost" if city in HIGH_COST_CITIES else "standard"


def mileage_rate(expense_date: date) -> float:
    """Return the per-mile rate in force on the expense date (POL-2.6).
    Raises:
        ValueError: If no rate band in policy.yaml covers the date.
    """
    for start, end, rate in MILEAGE_RATES:
        if start <= expense_date <= end:
            return rate
    raise ValueError(f"No mileage rate covers {expense_date}")


test_rng = random.Random(0)
sample_dates = [rand_date(WINDOW_START, WINDOW_END, test_rng) for _ in range(1_000)]
assert min(sample_dates) >= WINDOW_START, "Date before window"
assert max(sample_dates) <= WINDOW_END, "Date after window"
assert rand_date(TODAY, TODAY, test_rng) == TODAY, "One-day range failed"

assert city_tier("New York, NY") == "high_cost"
assert city_tier("San Antonio, TX") == "standard"

assert mileage_rate(date(2026, 6, 30)) == 0.725, "Last day of old rate"
assert mileage_rate(date(2026, 7, 1)) == 0.76, "first day of new rate"
try:
    mileage_rate(date(2025, 12, 31))
except ValueError:
    pass
else:
    raise AssertionError("2025 date should have no mileage rate")

assert money(12.345678) == 12.35
print(
    f"Helpers OK dates{min(sample_dates)} to {max(sample_dates)}, "
    f"mileage {mileage_rate(WINDOW_START)} -> {mileage_rate(WINDOW_END)}"
)

Helpers OK dates2026-05-01 to 2026-10-31, mileage 0.725 -> 0.76


In [ ]:
# 4a Lookup lists and the line-item template
CITIES = [
    "New York, NY",
    "San Francisco, CA",
    "Boston, MA",
    "Seattle, WA",
    "Washington, DC",
    "Los Angeles, CA",
    "Austin, TX",
    "Dallas, TX",
    "Chicago, IL",
    "Denver, CO",
    "Atlanta, GA",
    "Phoenix, AZ",
    "Nashville, TN",
    "San Antonio, TX",
]
CLIENTS = [
    "Acme Corp",
    "Globex",
    "Initech",
    "Umbrella Health",
    "Stark Logistics",
    "Wayne Financial",
    "Hooli",
    "Soylent Foods",
    "Cyberdyne Systems",
]
CLIENT_PEOPLE = [
    "Alex Morgan",
    "Jordan Lee",
    "Taylor Reed",
    "Casey Patel",
    "Riley Chen",
    "Morgan Diaz",
    "Avery Shah",
    "Quinn Foster",
    "Drew Kowalski",
    "Jamie Ito",
]
CONFERENCES = [
    "AWS re:Invent",
    "SaaStr Annual",
    "Data + AI Summit",
    "Dreamforce",
    "Content Marketing World",
    "PyData Global",
    "Gartner Data & Analytics Summit",
]
MEETING_TOPICS = [
    "renewal",
    "expansion",
    "pilot results",
    "Q4 roadmap",
    "contract terms",
]
HOME_OFFICE_ITEMS = [
    "External monitor",
    "Ergonomic chair",
    "Standing desk converter",
    "Docking station",
    "Webcam and headset",
    "Keyboard and mouse",
]
SUPPLY_ITEMS = [
    "Notebooks and pens",
    "Printer ink",
    "Whiteboard markers",
    "Shipping supplies",
    "Desk organizers",
]
RIDE_MERCHANTS = ["Uber", "Lyft", "Yellow Cab", "Airport Parking"]
RENTAL_MERCHANTS = ["Enterprise Rent-A-Car", "Hertz"]
MERCHANTS = {
    "Airfare": [
        "Delta Air Lines",
        "United Airlines",
        "American Airlines",
        "Southwest Airlines",
        "Alaska Airlines",
    ],
    "Lodging": [
        "Marriott",
        "Hilton",
        "Hyatt",
        "Courtyard by Marriott",
        "Hampton Inn",
        "Westin",
    ],
    "Meals": [
        "Starbucks",
        "Panera Bread",
        "Chipotle",
        "Hotel Restaurant",
        "Sweetgreen",
        "Local Diner",
    ],
    "Client Meals": [
        "Capital Grille",
        "Ruth's Chris Steak House",
        "Fogo de Chao",
        "Eddie V's",
        "Nobu",
        "The Mercer Kitchen",
    ],
    "Ground Transport": RIDE_MERCHANTS + RENTAL_MERCHANTS,
    "Conference": CONFERENCES,
    "Software": CATEGORIES["Software"]["approved_vendors"],
    "Home Office": ["Amazon", "Best Buy", "IKEA", "Staples", "Apple Store"],
    "Office Supplies": ["Staples", "Office Depot", "Amazon", "Target"],
}

RECEIPT_THRESHOLD = POLICY["submission"]["receipt_required_over"]
CLEAN_MAX_PCT = 1 - POLICY["escalation"]["near_cap_pct"] - 0.01


def blank_item(category: str, expense_date: date, city: str) -> dict[str, Any]:
    """Return an empty line item with every column set to a default.

    Every claim has the same 16 columns, even when a field does not
    apply (e.g. ``miles`` on  a hotel stay), so the final table is
    rectangular.
    """
    return {
        "expense_date": expense_date,
        "category": category,
        "merchant": "",
        "description": "",
        "business_purpose": "",
        "amount": 0.0,
        "city": city,
        "nights": None,
        "attendees": None,
        "attendee_names": "",
        "fare_class": "",
        "flight_hours": None,
        "rental_class": "",
        "miles": None,
        "preapproved": False,
        "receipt_attached": True,
    }


missing_merchants = set(CATEGORIES) - set(MERCHANTS)
assert missing_merchants == {"Mileage"}, f"No merchants for: {missing_merchants}"
assert HIGH_COST_CITIES <= set(CITIES), "A high-cost city is missing from CITIES"
assert round(CLEAN_MAX_PCT, 2) == 0.89, f"Unexpected CLEAN_MAX_PCT: {CLEAN_MAX_PCT}"
assert len(blank_item("Meals", TODAY, "Austin, TX")) == 16
print(f"Lookups OK: {len(CITIES)} cities, {len(MERCHANTS)} merchant lists")

Lookups OK: 14 cities, 9 merchant lists


In [ ]:
# 4b make_clean_item
def make_clean_item(
    category: str,
    expense_date: date,
    city: str,
    purpose: str,
    rand: random.Random,
    nights: int = 1,
) -> dict[str, Any]:
    """Build one in-policy line item for a category (POL-2).

    Amounts stay below ``CLEAN_MAX_PCT`` of every cap so a clean claim
    never looks "near cap" (POL-5) and never breaks a hard rule.

    Args:
        category: A key of ``CATEGORIES`` in policy.yaml.
        expense_date: Date the expense happened.
        city: City where it happened (sets the lodging tier).
        purpose: Business purpose for the trip or report.
        rand: Seeded random generator to draw from.
        nights: Hotel nights, used only for Lodging.

    Returns:
        A dict with the 16 line-item columns filled in.
    """
    if category not in CATEGORIES:
        raise ValueError(f"Unknown category: {category}")
    item = blank_item(category, expense_date, city)
    item["business_purpose"] = purpose
    rules = CATEGORIES[category]

    match category:
        case "Airfare":
            item["merchant"] = rand.choice(MERCHANTS[category])
            item["fare_class"] = "Economy"
            item["flight_hours"] = round(rand.uniform(1.2, 4.8), 1)
            item["amount"] = money(rand.uniform(180, 650))
            item["description"] = f"Round-trip economy airfare to {city}"
        case "Lodging":
            cap = rules["nightly_cap"][city_tier(city)]
            nightly = rand.uniform(0.55 * cap, CLEAN_MAX_PCT * cap)
            item["merchant"] = rand.choice(MERCHANTS[category])
            item["nights"] = nights
            item["amount"] = money(nightly * nights)
            item["description"] = f"Hotel, {nights} night(s) at ${nightly:,.2f}/night"
        case "Meals":
            cap = rules["daily_per_diem"]
            item["merchant"] = rand.choice(MERCHANTS[category])
            item["amount"] = money(rand.uniform(18, CLEAN_MAX_PCT * cap))
            item["description"] = "Traveler meals for the day"
        case "Client Meals":
            attendees = rand.randint(2, 5)
            client = rand.choice(CLIENTS)
            guests = rand.sample(CLIENT_PEOPLE, attendees - 1)
            per_person = rand.uniform(35, CLEAN_MAX_PCT * rules["per_person_cap"])
            names = [f"{guest} ({client})" for guest in guests] + ["employee"]
            item["merchant"] = rand.choice(MERCHANTS[category])
            item["attendees"] = attendees
            item["attendee_names"] = "; ".join(names)
            item["amount"] = money(per_person * attendees)
            item["description"] = f"Client dinner with {client}, {attendees} attendees"
            topic = rand.choice(MEETING_TOPICS)
            item["business_purpose"] = f"Dinner with {client} team to discuss {topic}"
        case "Ground Transport":
            if rand.random() < 0.25:
                rental_class = rand.choice(rules["allowed_rental_classes"])
                item["merchant"] = rand.choice(RENTAL_MERCHANTS)
                item["rental_class"] = rental_class
                item["amount"] = money(rand.uniform(120, 380))
                item["description"] = f"{rental_class} rental car"
            else:
                item["merchant"] = rand.choice(RIDE_MERCHANTS)
                item["amount"] = money(rand.uniform(12, 85))
                item["description"] = "Ride to client site / airport"
        case "Mileage":
            miles = rand.randint(20, 180)
            rate = mileage_rate(expense_date)
            item["merchant"] = "Personal vehicle"
            item["miles"] = miles
            item["amount"] = money(miles * rate)
            item["description"] = f"{miles} business miles at ${rate}/mile"
            item["receipt_attached"] = False
        case "Conference":
            conference = rand.choice(CONFERENCES)
            cap = rules["per_event_cap"]
            item["merchant"] = conference
            item["preapproved"] = True
            item["amount"] = money(rand.uniform(300, CLEAN_MAX_PCT * cap))
            item["description"] = f"{conference} registration"
            item["business_purpose"] = (
                f"Attend {conference} for training and customer networking"
            )
        case "Software":
            vendor = rand.choice(MERCHANTS[category])
            item["merchant"] = vendor
            item["amount"] = money(rand.uniform(10, 300))
            item["description"] = f"{vendor} monthly subscription"
        case "Home Office":
            cap = rules["per_purchase_cap"]
            item["merchant"] = rand.choice(MERCHANTS[category])
            item["amount"] = money(rand.uniform(40, CLEAN_MAX_PCT * cap))
            item["description"] = rand.choice(HOME_OFFICE_ITEMS)
        case "Office Supplies":
            cap = rules["per_purchase_cap"]
            item["merchant"] = rand.choice(MERCHANTS[category])
            item["amount"] = money(rand.uniform(8, CLEAN_MAX_PCT * cap))
            item["description"] = rand.choice(SUPPLY_ITEMS)

    # Pol-1.3: receipts always attached over the threshold; ~80% below it.
    if category != "Mileage":
        over_threshold = item["amount"] > RECEIPT_THRESHOLD
        item["receipt_attached"] = over_threshold or rand.random() < 0.8
    return item

In [ ]:
# 4c Tests for the generator
def clean_cap(item: dict[str, Any]) -> tuple[float, float] | None:
    """Return (value, cap) for capped categories, else None. Test helper."""
    rules = CATEGORIES[item["category"]]
    match item["category"]:
        case "Lodging":
            nightly = item["amount"] / item["nights"]
            return nightly, rules["nightly_cap"][city_tier(item["city"])]
        case "Meals":
            return item["amount"], rules["daily_per_diem"]
        case "Client Meals":
            return item["amount"] / item["attendees"], rules["per_person_cap"]
        case "Conference":
            return item["amount"], rules["per_event_cap"]
        case "Home Office" | "Office Supplies":
            return item["amount"], rules["per_purchase_cap"]
    return None


test_rng = random.Random(1)
checked = 0
for category in CATEGORIES:
    for _ in range(200):
        city = test_rng.choice(CITIES)
        item = make_clean_item(
            category, date(2026, 8, 3), city, "Test", test_rng, nights=3
        )
        checked += 1
        assert item["amount"] > 0, item
        capped = clean_cap(item)
        if capped:
            value, cap = capped
            assert value < CLEAN_MAX_PCT * cap + 0.01, f"Near/over cap: {item}"
        if category != "Mileage" and item["amount"] > RECEIPT_THRESHOLD:
            assert item["receipt_attached"], f"Missing receipt {item}"

first = make_clean_item("Meals", TODAY, "Austin, TX", "Test", random.Random(5))
second = make_clean_item("Meals", TODAY, "Austin, TX", "Test", random.Random(5))
assert first == second, "Same seed should give the same item"

try:
    make_clean_item("Yachts", TODAY, "Austin, TX", "Test", random.Random(5))
except ValueError:
    pass
else:
    raise AssertionError("Unknown category should raise ValueError")

print(f"Clean items OK: {checked} generated, 0 near or over a cap")

Clean items OK: 2000 generated, 0 near or over a cap
